### KDD · Fase 1: Selección

Selección de la base de datos y de las variables, y partición en grupos de entrenamiento y prueba.

- **Entrada:** `data/raw/penguins_size.csv`
- **Salidas:**
  - `kdd/data/interim/train.csv`, `kdd/data/interim/test.csv`: grupos de entrenamiento y prueba
  - `kdd/data/interim/selection_summary.csv`: resumen de la partición
- **Variable objetivo:** `species`

#### Librerías

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

#### Semilla

In [2]:
# Semilla para reproducibilidad. Puede modificarse o eliminarse.
SEED = 1

#### Rutas

Relativas a la carpeta del notebook.

In [3]:
RAW_PATH = Path("../../data/raw/penguins_size.csv")
INTERIM_DIR = Path("../data/interim")
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

#### Carga de datos

In [4]:
df = pd.read_csv(RAW_PATH)

print(f"Registros: {df.shape[0]} | Variables: {df.shape[1]}")
df.head()

Registros: 344 | Variables: 7


,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


#### Tipos de datos y valores faltantes

In [5]:
pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "no_nulos": df.notna().sum(),
    "faltantes": df.isna().sum(),
    "valores_distintos": df.nunique(),
})

,tipo,no_nulos,faltantes,valores_distintos
species,object,344,0,3
island,object,344,0,3
culmen_length_mm,float64,342,2,164
culmen_depth_mm,float64,342,2,80
flipper_length_mm,float64,342,2,55
body_mass_g,float64,342,2,94
sex,object,334,10,3


#### Distribución de la variable objetivo

In [6]:
pd.DataFrame({
    "registros": df["species"].value_counts(),
    "proporcion": df["species"].value_counts(normalize=True).round(3),
})

,registros,proporcion
species,,
Adelie,152,0.442
Gentoo,124,0.360
Chinstrap,68,0.198


#### Selección de variables

Las seis variables de `penguins_size.csv` se usan como predictoras.

In [7]:
TARGET = "species"
NUMERIC = ["culmen_length_mm", "culmen_depth_mm", "flipper_length_mm", "body_mass_g"]
CATEGORICAL = ["island", "sex"]
FEATURES = NUMERIC + CATEGORICAL

assert set(FEATURES + [TARGET]) == set(df.columns)
print(f"Predictoras: {FEATURES}")

Predictoras: ['culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm', 'body_mass_g', 'island', 'sex']


#### Identificador de registro

`row_id` guarda la posición original de cada fila. No se usa como variable predictora.

In [8]:
df = df.reset_index().rename(columns={"index": "row_id"})
df[["row_id", TARGET] + FEATURES].head()

,row_id,species,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,island,sex
0,0,Adelie,39.1,18.7,181.0,3750.0,Torgersen,MALE
1,1,Adelie,39.5,17.4,186.0,3800.0,Torgersen,FEMALE
2,2,Adelie,40.3,18.0,195.0,3250.0,Torgersen,FEMALE
3,3,Adelie,NaN,NaN,NaN,NaN,Torgersen,NaN
4,4,Adelie,36.7,19.3,193.0,3450.0,Torgersen,FEMALE


#### Partición 80/20 estratificada por `species`

In [9]:
TEST_SIZE = 0.20

train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    stratify=df[TARGET],
    random_state=SEED,
)

print(f"Entrenamiento: {len(train_df)} registros")
print(f"Prueba: {len(test_df)} registros")

Entrenamiento: 275 registros
Prueba: 69 registros


#### Verificación de la estratificación

In [10]:
summary = pd.DataFrame({
    "total_n": df[TARGET].value_counts(),
    "train_n": train_df[TARGET].value_counts(),
    "test_n": test_df[TARGET].value_counts(),
    "total_prop": df[TARGET].value_counts(normalize=True),
    "train_prop": train_df[TARGET].value_counts(normalize=True),
    "test_prop": test_df[TARGET].value_counts(normalize=True),
}).round(3)
summary.index.name = TARGET
summary

,total_n,train_n,test_n,total_prop,train_prop,test_prop
species,,,,,,
Adelie,152,122,30,0.442,0.444,0.435
Gentoo,124,99,25,0.360,0.360,0.362
Chinstrap,68,54,14,0.198,0.196,0.203


#### Guardado de resultados

In [11]:
train_df.to_csv(INTERIM_DIR / "train.csv", index=False)
test_df.to_csv(INTERIM_DIR / "test.csv", index=False)
summary.to_csv(INTERIM_DIR / "selection_summary.csv")

for path in ["train.csv", "test.csv", "selection_summary.csv"]:
    print(f"Guardado: {INTERIM_DIR / path}")

Guardado: ..\data\interim\train.csv
Guardado: ..\data\interim\test.csv
Guardado: ..\data\interim\selection_summary.csv
